<a href="https://colab.research.google.com/github/Samy-Annasri/backcasting-for-adversarial-attacks-on-time-series/blob/main/BATS_Pedestrian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset,TensorDataset
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import random
from datetime import datetime
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
seed = 888
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(seed)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def load_tsf(file_path, target_series_name="T9"):
    with open(file_path, "r", encoding="ISO-8859-1") as file:
        lines = file.readlines()

    data_started = False
    all_series = []

    for line in lines:
        line = line.strip()
        if line == "@data":
            data_started = True
            continue
        if not data_started or line.startswith("@"):
            continue

        parts = line.split(":", 2)
        if len(parts) != 3:
            continue

        series_name = parts[0]
        start_timestamp = parts[1]
        series_values = list(map(float, parts[2].split(",")))

        if series_name == target_series_name:
            all_series.append({
                "series_name": series_name,
                "timestamp": start_timestamp,
                "series": series_values
            })
            break

    return all_series

def fix_timestamp(ts_str):
    date_part, time_part = ts_str.strip().split(" ")
    time_parts = time_part.split("-")

    if len(time_parts) == 3:
        hour, minute, _ = time_parts
    else:
        hour, minute = time_parts[0], time_parts[1] if len(time_parts) > 1 else "00"

    return f"{date_part} {hour}:{minute}:00"


def load_tsf_flat(file_path, series_name="T9", freq="H"):
    data = load_tsf(file_path, target_series_name=series_name)
    if not data:
        raise ValueError(f"Série {series_name} non trouvée dans le fichier.")
    series = data[0]

    start_str = fix_timestamp(series["timestamp"])
    start_date = datetime.strptime(start_str, "%Y-%m-%d %H:%M:%S")


    return pd.DataFrame({
        "date": pd.date_range(start=start_date, periods=len(series["series"]), freq=freq),
        "value": series["series"]
    })


In [ ]:
df = load_tsf_flat("data/pedestrian_counts_dataset.tsf", series_name="T9", freq="h")
print(df.head())
print(df.tail())

In [ ]:
print(df["value"].describe())

import matplotlib.pyplot as plt
df.plot(x="date", y="value", figsize=(12,5), title="pedestrian_counts")
plt.show()


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def prepare_pedestrian_dataset(
    df,
    sequence_length=30,
    batch_size=64,
    split_ratio=0.8,
    reverse=False
):
    serie = df["value"].values.astype(float)

    serie_min = np.min(serie)
    serie_max = np.max(serie)
    serie_norm = (serie - serie_min) / (serie_max - serie_min + 1e-8)

    X, Y, used_dates = [], [], []

    if not reverse:
        for i in range(len(serie_norm) - sequence_length):
            X.append(serie_norm[i:i + sequence_length])
            Y.append(serie_norm[i + sequence_length])
            used_dates.append(df["date"].iloc[i + sequence_length])
    else:
        for i in range(sequence_length, len(serie_norm)):
            seq = serie_norm[i-sequence_length:i][::-1]
            target_idx = i - sequence_length
            X.append(seq)
            Y.append(serie_norm[target_idx])
            used_dates.append(df["date"].iloc[target_idx])

    X = np.array(X)
    Y = np.array(Y)

    split = int(len(X) * split_ratio)
    X_train, X_test = X[:split], X[split:]
    Y_train, Y_test = Y[:split], Y[split:]

    dates_train = used_dates[:split]
    dates_test = used_dates[split:]

    X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
    Y_train = torch.tensor(Y_train, dtype=torch.float32).unsqueeze(-1)
    X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)
    Y_test = torch.tensor(Y_test, dtype=torch.float32).unsqueeze(-1)

    train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(TensorDataset(X_test, Y_test), batch_size=batch_size, drop_last=True)

    return {
        'train_loader': train_loader,
        'test_loader': test_loader,
        'train_size': len(X_train),
        'test_size': len(X_test),
        'min_max': (serie_min, serie_max),
        'dates_train': dates_train,
        'dates_test': dates_test,
        'X_all': torch.tensor(X, dtype=torch.float32).unsqueeze(-1),
        'Y_all': torch.tensor(Y, dtype=torch.float32).unsqueeze(-1)
    }


In [ ]:
sequence_length = 30
result = prepare_pedestrian_dataset(df, sequence_length)

train_loader = result['train_loader']
test_loader = result['test_loader']
train_size = result['train_size']
serie_min, serie_max = result['min_max']
dates = result['dates_train']
dates_test = result['dates_test']
print(dates)

In [ ]:
#name model_elec but work also with solar/birth btw
from utils.elec.train_model_elec import train_models_elec

In [ ]:
from models.lstm import SimpleLSTM
model = SimpleLSTM(input_size=1, hidden_size=64, output_size=1, num_layers=2)
model.to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

train_models_elec(model, loss_fn, optimizer, num_epochs, train_loader,device)

In [ ]:
from utils.birth.evaluate_model_births import evaluate_model_births

In [ ]:
results = evaluate_model_births(model, test_loader, dates_test, train_size)

real_values = results['real_values']
predicted_values = results['predicted_values']
test_dates = results['test_dates']

In [ ]:
# Computes scalar similarity (cosine similarity) between true and predicted values.
# Higher values indicate that adversarial predictions remain directionally aligned
# with the true values, suggesting stealthy and rational attacks!
def scalar_similarity(y_true, y_pred):
    numerator = np.dot(y_true, y_pred)
    denominator = np.linalg.norm(y_true) * np.linalg.norm(y_pred)
    if denominator == 0:
        return 0.0
    return numerator / denominator

In [ ]:
# Creation of the tab result for plotting adversial attack result
models = ['LSTM','RNN','GRU']
metrics = ['MAE', "RMSE", 'SIM']

row_index = pd.MultiIndex.from_product([models, metrics], names=['Model', 'Metric'])

attacks = ['NA','PAST','REV']
epsilons = {
    'NA': [0],
    'PAST':[0],
    'REV':[0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2],
}

col_tuples = []
for atk, eps_list in epsilons.items():
    for eps in eps_list:
        col_tuples.append((atk, f"{eps:.2f}"))

col_index = pd.MultiIndex.from_tuples(col_tuples, names=['Attack', 'ε'])

res_tab = pd.DataFrame(index=row_index, columns=col_index, dtype=float)

print(res_tab)

In [ ]:
from utils.solar.log_and_plot_solar import log_and_plot_solar

In [ ]:
def log_and_plot_pedestrian(true_values, predictions,
                             model_name, attack_name, epsilon,
                             res_tab, similarity_fn,
                             save_path=None, save_png=False):

    mae = mean_absolute_error(true_values, predictions)
    sim = similarity_fn(true_values, predictions)
    rmse = np.sqrt(mean_squared_error(true_values, predictions))

    eps_str = f"{epsilon:.2f}"

    res_tab.loc[(model_name, 'MAE'), (attack_name, eps_str)] = mae
    res_tab.loc[(model_name, 'SIM'), (attack_name, eps_str)] = sim
    res_tab.loc[(model_name, 'RMSE'), (attack_name, eps_str)] = rmse

    print(f"{model_name} | {attack_name} – Epsilon {eps_str} – MAE: {mae:.4f} | SIM: {sim:.4f} | RMSE: {rmse:.4f}")

    x = list(range(len(true_values)))

    plt.figure(figsize=(12, 6))
    plt.plot(x, true_values, label='Valeurs réelles', color='blue')
    plt.plot(x, predictions, label='Prédictions', color='red', linestyle='--')
    plt.xlabel('Heures')
    plt.ylabel('Comptage de piétons')
    plt.title(f"Comptage de piétons – {model_name} ({attack_name}, eps={eps_str})")
    plt.legend()
    plt.tight_layout()

    if save_png and save_path:
        os.makedirs(save_path, exist_ok=True)
        filename = f"{model_name.lower()}_{attack_name.lower()}_eps_{epsilon:.2f}.png"
        plt.savefig(os.path.join(save_path, filename))

    plt.show()


In [ ]:
# Dénormalisation
real_values_denorm = results['real_values'] * (serie_max - serie_min) + serie_min
predicted_values_denorm = results['predicted_values'] * (serie_max - serie_min) + serie_min

days_plot = 400

log_and_plot_pedestrian(
    true_values=real_values_denorm[-days_plot:],
    predictions=predicted_values_denorm[-days_plot:],
    model_name='LSTM',
    attack_name='NA',
    epsilon=0.0,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    save_path=f"results/pedestrian/SEED_{seed}/plots/NA/0",
    save_png=True
)


In [ ]:
result_rev = prepare_pedestrian_dataset(df, sequence_length,reverse=True)

train_loader_rev = result_rev['train_loader']
test_loader_rev = result_rev['test_loader']
train_size_rev = result_rev['train_size']
serie_min_rev, serie_max_rev = result_rev['min_max']
dates_rev = result_rev['dates_test']

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

axs[0].plot(result['X_all'][100].squeeze(), label='Normal')
axs[0].set_title("Normal Sequence")
axs[0].legend()
axs[0].grid(True)

axs[1].plot(result_rev['X_all'][100].squeeze(), label='Reverse', color='orange')
axs[1].set_title("Reversed Sequence")
axs[1].legend()
axs[1].grid(True)

plt.suptitle("Visual Comparison of Input Sequences")
plt.tight_layout()
plt.show()

In [ ]:
model_rev = SimpleLSTM(input_size=1, hidden_size=64, output_size=1, num_layers=2)
optimizer_rev = torch.optim.Adam(model_rev.parameters(), lr=0.001)
model_rev.to(device)
train_models_elec(model_rev, loss_fn, optimizer_rev, num_epochs, train_loader_rev,device)

In [ ]:
results_rev = evaluate_model_births(model_rev, test_loader_rev, dates_rev, train_size_rev)

real_values_rev = results_rev['real_values']
predicted_values_rev = results_rev['predicted_values']
test_dates_rev = results_rev['test_dates']

In [ ]:
# Denormalize
true_values_denorm_rev = real_values_rev * (serie_max_rev - serie_min_rev) + serie_min_rev
predictions_denorm_rev = predicted_values_rev * (serie_max_rev - serie_min_rev) + serie_min_rev

log_and_plot_pedestrian(
    true_values=true_values_denorm_rev[-days_plot:],
    predictions=predictions_denorm_rev[-days_plot:],
    model_name='LSTM',
    attack_name='PAST',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    save_path=f"results/pedestrian/SEED_{seed}/plots/PAST/0",
    save_png=True
)

In [ ]:
from attack.rev import reverse_forecast_attack

epsilons = [0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2]
results = reverse_forecast_attack(
    model_rev, model,
    test_loader_rev, test_loader,
    epsilons,
    serie_min_rev, serie_max_rev
)

for esp, (true_vals, preds) in results.items():
  log_and_plot_pedestrian(
      true_values=true_vals[-days_plot:],
      predictions=preds[-days_plot:],
      model_name='LSTM',
      attack_name='REV',
      epsilon=esp,
      res_tab=res_tab,
      similarity_fn=scalar_similarity,
      save_path=f"results/pedestrian/SEED_{seed}/plots/REV/{esp}",
      save_png=True
  )

In [ ]:
display(res_tab)

In [ ]:
from models.rnn import SimpleRNN
model_rnn = SimpleRNN(input_size=1, hidden_size=64, output_size=1, num_layers=2)
model_rnn.to(device)
optimizer = torch.optim.Adam(model_rnn.parameters(), lr=0.001)
train_models_elec(model_rnn, loss_fn, optimizer, num_epochs, train_loader,device)

In [ ]:
results_rnn = evaluate_model_births(model_rnn, test_loader, dates, train_size)

real_values_rnn = results_rnn['real_values']
predicted_values_rnn = results_rnn['predicted_values']
test_dates_rnn = results_rnn['test_dates']

true_values_denorm_rnn = real_values_rnn * (serie_max - serie_min) + serie_min
predictions_denorm_rnn = predicted_values_rnn * (serie_max - serie_min) + serie_min

log_and_plot_pedestrian(
    true_values=true_values_denorm_rnn[-days_plot:],
    predictions=predictions_denorm_rnn[-days_plot:],
    model_name='RNN',
    attack_name='NA',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    save_path=f"results/pedestrian/SEED_{seed}/plots/PAST/0",
    save_png=True
)

In [ ]:
model_rnn_rev = SimpleRNN(input_size=1, hidden_size=64, output_size=1, num_layers=2)
model_rnn_rev.to(device)
optimizer = torch.optim.Adam(model_rnn_rev.parameters(), lr=0.001)
train_models_elec(model_rnn_rev, loss_fn, optimizer, num_epochs, train_loader_rev,device)

In [ ]:
results_rev_rnn = evaluate_model_births(model_rnn_rev, test_loader_rev, dates_rev, train_size_rev)

real_values_rev_rnn = results_rev_rnn['real_values']
predicted_values_rev_rnn = results_rev_rnn['predicted_values']
test_dates_rev_rnn = results_rev_rnn['test_dates']

true_values_denorm_rev_rnn = real_values_rev_rnn * (serie_max_rev - serie_min_rev) + serie_min_rev
predictions_denorm_rev_rnn = predicted_values_rev_rnn * (serie_max_rev - serie_min_rev) + serie_min_rev

log_and_plot_pedestrian(
    true_values=true_values_denorm_rev_rnn[-days_plot:],
    predictions=predictions_denorm_rev_rnn[-days_plot:],
    model_name='RNN',
    attack_name='PAST',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    save_path=f"results/pedestrian/SEED_{seed}/plots/PAST/0",
    save_png=True
)

In [ ]:
results = reverse_forecast_attack(
    model_rnn_rev, model_rnn,
    test_loader_rev, test_loader,
    epsilons,
    serie_min_rev, serie_max_rev
)

for esp, (true_vals, preds) in results.items():
  log_and_plot_pedestrian(
      true_values=true_vals[-days_plot:],
      predictions=preds[-days_plot:],
      model_name='RNN',
      attack_name='REV',
      epsilon=esp,
      res_tab=res_tab,
      similarity_fn=scalar_similarity,
      save_path=f"results/pedestrian/SEED_{seed}/plots/REV/{esp}",
      save_png=True
  )

In [ ]:
from models.gru import SimpleGRU
model_gru = SimpleGRU(input_size=1, hidden_size=64, output_size=1, num_layers=2)
model_gru.to(device)
optimizer = torch.optim.Adam(model_gru.parameters(), lr=0.001)
train_models_elec(model_gru, loss_fn, optimizer, num_epochs, train_loader,device)

In [ ]:
results_gru = evaluate_model_births(model_gru, test_loader, dates, train_size)

real_values_gru = results_gru['real_values']
predicted_values_gru = results_gru['predicted_values']
test_dates_gru = results_gru['test_dates']

true_values_denorm_gru = real_values_gru * (serie_max - serie_min) + serie_min
predictions_denorm_gru = predicted_values_gru * (serie_max - serie_min) + serie_min

log_and_plot_pedestrian(
    true_values=true_values_denorm_gru[-days_plot:],
    predictions=predictions_denorm_gru[-days_plot:],
    model_name='GRU',
    attack_name='NA',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    save_path=f"results/pedestrian/SEED_{seed}/plots/PAST/0",
    save_png=True
)

In [ ]:
model_gru_rev = SimpleGRU(input_size=1, hidden_size=64, output_size=1, num_layers=2)
model_gru_rev.to(device)
optimizer = torch.optim.Adam(model_gru_rev.parameters(), lr=0.001)
train_models_elec(model_gru_rev, loss_fn, optimizer, num_epochs, train_loader_rev,device)

In [ ]:
results_rev_gru = evaluate_model_births(model_gru_rev, test_loader_rev, dates_rev, train_size_rev)

real_values_rev_gru = results_rev_gru['real_values']
predicted_values_rev_gru = results_rev_gru['predicted_values']
test_dates_rev_gru = results_rev_gru['test_dates']

true_values_denorm_rev_gru = real_values_rev_gru * (serie_max_rev - serie_min_rev) + serie_min_rev
predictions_denorm_rev_gru = predicted_values_rev_gru * (serie_max_rev - serie_min_rev) + serie_min_rev

log_and_plot_pedestrian(
    true_values=true_values_denorm_rev_gru[-days_plot:],
    predictions=predictions_denorm_rev_gru[-days_plot:],
    model_name='GRU',
    attack_name='PAST',
    epsilon=0.00,
    res_tab=res_tab,
    similarity_fn=scalar_similarity,
    save_path=f"results/pedestrian/SEED_{seed}/plots/PAST/0",
    save_png=True
)

In [ ]:
results = reverse_forecast_attack(
    model_gru_rev, model_rnn,
    test_loader_rev, test_loader,
    epsilons,
    serie_min_rev, serie_max_rev
)

for esp, (true_vals, preds) in results.items():
  log_and_plot_pedestrian(
      true_values=true_vals[-days_plot:],
      predictions=preds[-days_plot:],
      model_name='GRU',
      attack_name='REV',
      epsilon=esp,
      res_tab=res_tab,
      similarity_fn=scalar_similarity,
      save_path=f"results/pedestrian/SEED_{seed}/plots/REV/{esp}",
      save_png=True
  )

In [ ]:
display(res_tab)

In [ ]:
from attack.fgsm import fgsm_attack

In [ ]:
# Creation of the tab result for plotting adversial attack result
models = ['LSTM','RNN','GRU']
metrics = ['MAE', "RMSE", 'SIM']

row_index = pd.MultiIndex.from_product([models, metrics], names=['Model', 'Metric'])

attacks = ['FGSM']
epsilons_attack = {
    'FGSM':[0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2],
}

col_tuples = []
for atk, eps_list in epsilons_attack.items():
    for eps in eps_list:
        col_tuples.append((atk, f"{eps:.2f}"))

col_index = pd.MultiIndex.from_tuples(col_tuples, names=['Attack', 'ε'])

res_tab_fgsm = pd.DataFrame(index=row_index, columns=col_index, dtype=float)

print(res_tab_fgsm)

In [ ]:
epsilons_fgsm = [0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2]

for eps in epsilons_fgsm:
    true_vals_fgsm, preds_fgsm = fgsm_attack(
        model_gru, test_loader, loss_fn, eps, serie_min_rev, serie_max_rev
    )

    log_and_plot_pedestrian(
        true_values=true_vals_fgsm[-days_plot:],
        predictions=preds_fgsm[-days_plot:],
        model_name='GRU',
        attack_name='FGSM',
        epsilon=eps,
        res_tab=res_tab_fgsm,
        similarity_fn=scalar_similarity,
        save_path=f"results/pedestrian/SEED_{seed}/plots/FGSM/{eps}",
        save_png=True
    )

    true_vals_fgsm, preds_fgsm = fgsm_attack(
        model, test_loader, loss_fn, eps, serie_min_rev, serie_max_rev
    )

    log_and_plot_pedestrian(
        true_values=true_vals_fgsm[-days_plot:],
        predictions=preds_fgsm[-days_plot:],
        model_name='LSTM',
        attack_name='FGSM',
        epsilon=eps,
        res_tab=res_tab_fgsm,
        similarity_fn=scalar_similarity,
        save_path=f"results/pedestrian/SEED_{seed}/plots/FGSM/{eps}",
        save_png=True
    )

    true_vals_fgsm, preds_fgsm = fgsm_attack(
        model_rnn, test_loader, loss_fn, eps, serie_min_rev, serie_max_rev
    )

    log_and_plot_pedestrian(
        true_values=true_vals_fgsm[-days_plot:],
        predictions=preds_fgsm[-days_plot:],
        model_name='RNN',
        attack_name='FGSM',
        epsilon=eps,
        res_tab=res_tab_fgsm,
        similarity_fn=scalar_similarity,
        save_path=f"results/pedestrian/SEED_{seed}/plots/FGSM/{eps}",
        save_png=True
    )


In [ ]:
display(res_tab_fgsm)

In [ ]:
from attack.rev_bim import reverse_forecast_attack_bim

In [ ]:
# Creation of the tab result for plotting adversial attack result
models = ['LSTM','RNN','GRU']
metrics = ['MAE', "RMSE", 'SIM']

row_index = pd.MultiIndex.from_product([models, metrics], names=['Model', 'Metric'])

attacks = ['REV_BIM']
epsilons = {
    'REV_BIM':[0.01,0.02,0.03,0.04, 0.05, 0.075, 0.1, 0.2],
}

col_tuples = []
for atk, eps_list in epsilons.items():
    for eps in eps_list:
        col_tuples.append((atk, f"{eps:.2f}"))

col_index = pd.MultiIndex.from_tuples(col_tuples, names=['Attack', 'ε'])

res_tab_bim = pd.DataFrame(index=row_index, columns=col_index, dtype=float)

print(res_tab_bim)

In [ ]:
alpha = 0.01
epsilons = [0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.1, 0.2]

bim_attacks = [
    {
        "model_rev": model_gru_rev,
        "model_target": model_gru,
        "model_name": "GRU"
    },
    {
        "model_rev": model_rev,
        "model_target": model,
        "model_name": "LSTM"
    },
    {
        "model_rev": model_rnn_rev,
        "model_target": model_rnn,
        "model_name": "RNN"
    }
]

for attack in bim_attacks:
    for eps in epsilons:
        num_iter = int(eps / alpha)

        results_bim = reverse_forecast_attack_bim(
            model_rev=attack["model_rev"],
            model_normal=attack["model_target"],
            test_loader_rev=test_loader_rev,
            test_loader=test_loader,
            epsilons=[eps],
            price_min=serie_min_rev,
            price_max=serie_max_rev,
            alpha=alpha,
            num_iter=num_iter
        )

        for eps_val, (true_vals, preds) in results_bim.items():
            log_and_plot_pedestrian(
              true_values=true_vals[-days_plot:],
              predictions=preds[-days_plot:],
              model_name=attack["model_name"],
              attack_name="REV_BIM",
              epsilon=eps_val,
              res_tab=res_tab_bim,
              similarity_fn=scalar_similarity,
              save_path=f"results/pedestrian/SEED_{seed}/plots/REV_BIM/{eps_val}",
              save_png=True
            )


In [ ]:
display(res_tab_bim)

In [ ]:
display(res_tab)

In [ ]:
model_surrogate = SimpleLSTM(input_size=1, hidden_size=64, output_size=1, num_layers=2)
optimizer_rev = torch.optim.Adam(model_surrogate.parameters(), lr=0.001)
model_surrogate.to(device)
train_models_elec(model_surrogate, loss_fn, optimizer_rev, num_epochs, train_loader,device)

In [ ]:
model_rnn_surrogate = SimpleRNN(input_size=1, hidden_size=64, output_size=1, num_layers=2)
optimizer_rev = torch.optim.Adam(model_rnn_surrogate.parameters(), lr=0.001)
model_rnn_surrogate.to(device)
train_models_elec(model_rnn_surrogate, loss_fn, optimizer_rev, num_epochs, train_loader,device)

In [ ]:
model_gru_surrogate = SimpleGRU(input_size=1, hidden_size=64, output_size=1, num_layers=2)
optimizer_rev = torch.optim.Adam(model_gru_surrogate.parameters(), lr=0.001)
model_gru_surrogate.to(device)
train_models_elec(model_gru_surrogate, loss_fn, optimizer_rev, num_epochs, train_loader,device)

In [ ]:
from attack.fgsm_surrogate import fgsm_surrogate_attack

In [ ]:
for eps in epsilons:
    true_vals_surro, preds_surro = fgsm_surrogate_attack(
        model_gru_surrogate,
        model_gru,
        test_loader,
        eps,
        serie_min_rev, serie_max_rev
    )

    log_and_plot_pedestrian(
        true_values=true_vals_surro[-days_plot:],
        predictions=preds_surro[-days_plot:],
        model_name='GRU',
        attack_name='FGSM_SURRO',
        epsilon=eps,
        res_tab=res_tab_bim,
        similarity_fn=scalar_similarity,
        save_path=f"results/pedestrian/SEED_{seed}/plots/FGSM_SURRO/{eps}",
        save_png=True
    )

    true_vals_surro, preds_surro = fgsm_surrogate_attack(
        model_rnn_surrogate,
        model_rnn,
        test_loader,
        eps,
        serie_min_rev, serie_max_rev
    )

    log_and_plot_pedestrian(
        true_values=true_vals_surro[-days_plot:],
        predictions=preds_surro[-days_plot:],
        model_name='RNN',
        attack_name='FGSM_SURRO',
        epsilon=eps,
        res_tab=res_tab_bim,
        similarity_fn=scalar_similarity,
        save_path=f"results/pedestrian/SEED_{seed}/plots/FGSM_SURRO/{eps}",
        save_png=True
    )

    true_vals_surro, preds_surro = fgsm_surrogate_attack(
        model_surrogate,
        model,
        test_loader,
        eps,
        serie_min_rev, serie_max_rev
    )

    log_and_plot_pedestrian(
        true_values=true_vals_surro[-days_plot:],
        predictions=preds_surro[-days_plot:],
        model_name='LSTM',
        attack_name='FGSM_SURRO',
        epsilon=eps,
        res_tab=res_tab_bim,
        similarity_fn=scalar_similarity,
        save_path=f"results/pedestrian/SEED_{seed}/plots/FGSM_SURRO/{eps}",
        save_png=True
    )

In [ ]:
display(res_tab_bim)

In [ ]:
# Creation of the tab result for plotting adversial attack result
models = ['LSTM','RNN','GRU']
metrics = ['MAE', "RMSE", 'SIM']

row_index = pd.MultiIndex.from_product([models, metrics], names=['Model', 'Metric'])

attacks = ['BOUNDARY']
epsilons_attack = {
    'BOUNDARY':[0.01,0.02,0.03,0.04, 0.05, 0.075, 0.1, 0.2],
}

col_tuples = []
for atk, eps_list in epsilons_attack.items():
    for eps in eps_list:
        col_tuples.append((atk, f"{eps:.2f}"))

col_index = pd.MultiIndex.from_tuples(col_tuples, names=['Attack', 'ε'])

res_tab_boundary = pd.DataFrame(index=row_index, columns=col_index, dtype=float)

print(res_tab_boundary)

In [ ]:
from attack.boundary import boundary_attack

In [ ]:
for eps in epsilons:
  delta = eps * 0.5
  eta = eps * 0.25
  num_iter = int(20 + 200 * eps)
  true_vals_bondary, preds_bondary = boundary_attack(
      model_gru,
      test_loader,
      eps,
      serie_min_rev, serie_max_rev,
      num_iter,delta,eta
  )

  log_and_plot_pedestrian(
      true_values=true_vals_bondary[-days_plot:],
      predictions=preds_bondary[-days_plot:],
      model_name='GRU',
      attack_name='BOUNDARY',
      epsilon=eps,
      res_tab=res_tab_boundary,
      similarity_fn=scalar_similarity,
      save_path=f"results/pedestrian/SEED_{seed}/plots/BOUNDARY/{eps}",
      save_png=True
  )

  true_vals_bondary, preds_bondary = boundary_attack(
      model,
      test_loader,
      eps,
      serie_min_rev, serie_max_rev,
      num_iter,delta,eta
  )

  log_and_plot_pedestrian(
      true_values=true_vals_bondary[-days_plot:],
      predictions=preds_bondary[-days_plot:],
      model_name='LSTM',
      attack_name='BOUNDARY',
      epsilon=eps,
      res_tab=res_tab_boundary,
      similarity_fn=scalar_similarity,
      save_path=f"results/pedestrian/SEED_{seed}/plots/BOUNDARY/{eps}",
      save_png=True
  )

  true_vals_bondary, preds_bondary = boundary_attack(
      model_rnn,
      test_loader,
      eps,
      serie_min_rev, serie_max_rev,
      num_iter,delta,eta
  )

  log_and_plot_pedestrian(
      true_values=true_vals_bondary[-days_plot:],
      predictions=preds_bondary[-days_plot:],
      model_name='RNN',
      attack_name='BOUNDARY',
      epsilon=eps,
      res_tab=res_tab_boundary,
      similarity_fn=scalar_similarity,
      save_path=f"results/pedestrian/SEED_{seed}/plots/BOUNDARY/{eps}",
      save_png=True
  )

In [ ]:
display(res_tab_boundary)

In [ ]:
result = pd.concat([res_tab, res_tab_bim,res_tab_fgsm,res_tab_boundary], axis=1)

# Affichage du résultat
display(result)
result.to_excel(f"results/pedestrian/SEED_{seed}/result.xlsx")